<a href="https://colab.research.google.com/github/ismoil27/deep_vision_mastery/blob/main/DocumentReader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image
from collections import defaultdict, Counter

import random
import os

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cuda


In [10]:
CLASSES = ['Receipts', 'Others']
CLASS_TO_IDX = {'Receipts': 0, 'Others': 1}
NUM_CLASSES = 2

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.299, 0.224, 0.225]
    )
])


class DocumentDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        for folder in os.listdir(root_dir):
            folder_path = os.path.join(root_dir, folder)

            if not os.path.isdir(folder_path):
                continue

            if folder == "Receipts":
                label = 0
            else:
                label = 1

            for img_name in os.listdir(folder_path):
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    img_path = os.path.join(folder_path, img_name)
                    self.samples.append((img_path, label))


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


In [3]:
data_dir = '/content/drive/MyDrive/DCV'
dataset = DocumentDataset(data_dir, transform=transform)

print('Total samples BEFORE balance:', len(dataset))

Total samples BEFORE balance: 4372


In [11]:
# Class distribution
labels = [label for _, label in dataset.samples]
count = Counter(labels)

print('\n Before balance:')
print(f'Receipts: {count[0]}')
print(f'Others: {count[1]}')


class_samples = defaultdict(list)

for path, label in dataset.samples:
  class_samples[label].append((path, label))

min_count = min(len(class_samples[0]), len(class_samples[1]))
print('\n Using per class:', min_count)

balanced_samples = []

for label in class_samples:
  random.shuffle(class_samples[label])
  balanced_samples.extend(class_samples[label][:min_count])

dataset.samples = balanced_samples
dataset.targets = [label for _, label in balanced_samples]

print('Total samples AFTER balance:', len(dataset))


 Before balance:
Receipts: 438
Others: 438

 Using per class: 438
Total samples AFTER balance: 876


In [12]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [17]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 230MB/s]
